In [52]:
# =====================================================================
# CELL 1 — CONFIGURATION
# =====================================================================
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

ROOT = next(folder for folder in (Path.cwd(), Path.cwd().parent) if (folder / "data").is_dir())
DATA_DIR = ROOT / "data" / "cleaned"

Q3_START     = pd.Timestamp("2026-07-01")
Q3_END       = pd.Timestamp("2026-10-01")
Q3_DAYS      = (Q3_END - Q3_START).days
EXTRACT_DATE = pd.Timestamp("2026-11-05")

PAID_CHANNELS = ["Meta", "Google Ads", "TikTok"]
EXPECTED_CHANNELS = PAID_CHANNELS
EXPECTED_ROWS_PER_DAY = 3
EXPECTED_TOTAL_ROWS = Q3_DAYS * EXPECTED_ROWS_PER_DAY

DEC_BUDGET = 150_000
DEC_DAYS = 31

# Three split scenarios used throughout
CPI_OPTIMISED_SHARES = {"Meta": 0.45, "Google Ads": 0.35, "TikTok": 0.20}  # Phase 3 (CPI-only)
FINAL_SHARES         = {"Meta": 0.45, "Google Ads": 0.40, "TikTok": 0.15}  # Phase 5 (activated-based)

SATURATION_FLAT_PCT     = 5.0
SATURATION_MODERATE_PCT = 10.0


SOURCE_TO_CHANNEL = {
    "meta": "Meta", "facebook ads": "Meta", "facebook": "Meta", "fb": "Meta",
    "instagram": "Meta", "ig": "Meta",
    "google": "Google Ads", "google ads": "Google Ads", "adwords": "Google Ads", "gads": "Google Ads",
    "tiktok": "TikTok", "tt": "TikTok",
    "referral": "Referral", "ref": "Referral", "invite": "Referral",
    "organic": "Organic", "(direct)": "Direct",
}

OTHER_CHANNELS = ["Organic", "Referral", "Direct"]


pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
print("✓ Config loaded")

✓ Config loaded


In [53]:
# =====================================================================
# PHASE 1 — CPI from marketing_spend (platform's reported view)
# =====================================================================
spend = pd.read_csv(DATA_DIR / "marketing_spend.csv", parse_dates=["date"])

# Validation
assert len(spend) == EXPECTED_TOTAL_ROWS, f"Row count {len(spend)}"
assert spend.duplicated(["date", "channel"]).sum() == 0, "Dup (date, channel)"
assert (spend.groupby("date").size() == EXPECTED_ROWS_PER_DAY).all(), "Day row count"
assert set(spend.channel) == set(EXPECTED_CHANNELS), "Unexpected channels"

q3_spend = spend.query("date >= @Q3_START and date < @Q3_END").copy()

cpi = (q3_spend.groupby("channel", as_index=False)
       .agg(spend_egp=("spend_egp", "sum"),
            reported_installs=("platform_reported_installs", "sum"))
       .assign(cpi=lambda d: (d.spend_egp / d.reported_installs).round(2)))

total_spend_q3    = q3_spend.spend_egp.sum()
total_installs_q3 = q3_spend.platform_reported_installs.sum()

print(f"Total Q3 spend    : {total_spend_q3:,.2f} EGP")
print(f"Total Q3 installs : {total_installs_q3:,}")
print(f"Blended CPI       : {total_spend_q3/total_installs_q3:.2f} EGP\n")
print("── CPI per channel ──")
display(cpi.set_index("channel"))

CPI_PER_CHANNEL = cpi.set_index("channel")["cpi"].to_dict()

# ── Recalculate CPI after crediting Google Ads with all unattributed signups ──
# Assume each unattributed signup represents one additional Google Ads install.

Total Q3 spend    : 356,538.71 EGP
Total Q3 installs : 6,343
Blended CPI       : 56.21 EGP

── CPI per channel ──


,spend_egp,reported_installs,cpi
channel,,,
Google Ads,"168,517.02",1803,93.46
Meta,"145,369.33",3355,43.33
TikTok,"42,652.36",1185,35.99


In [54]:
# =====================================================================
# PHASE 2 — Cost per signup (reconciled to internal users table)
# =====================================================================
users = pd.read_csv(DATA_DIR / "users.csv", parse_dates=["signup_ts", "kyc_verified_ts"])
source = users["utm_source_normalized"]
users["channel"] = source.map(SOURCE_TO_CHANNEL).fillna("Unknown")
users.loc[source.isna(), "channel"] = "Unattributed"
assert (users.channel == "Unknown").sum() == 0, "Unmapped source"

users_q3 = users.query("signup_ts >= @Q3_START and signup_ts < @Q3_END")

paid_signups = (users_q3[users_q3.channel.isin(PAID_CHANNELS)]
                .groupby("channel").size()
                .reindex(PAID_CHANNELS).fillna(0).astype(int))
n_unattr = int((users_q3.channel == "Unattributed").sum())

other_channels = ~users_q3["channel"].isin(PAID_CHANNELS + ["Unattributed"])
n_other = int(other_channels.sum())

spend_by_channel = cpi.set_index("channel").loc[PAID_CHANNELS, "spend_egp"]

cpsu = pd.DataFrame({
    "spend_egp": spend_by_channel,
    "paid_signups": paid_signups,
    "cost_per_signup": (spend_by_channel / paid_signups).round(2),
})

print(f"Clean users: {len(users):,}")
print()

print("── Cost per signup ──")
display(cpsu)

print()
print(f"Other channels     : {n_other:,}, {OTHER_CHANNELS}")
print(f"Unattributed       : {n_unattr:,}\n")

COST_PER_SIGNUP = cpsu["cost_per_signup"].to_dict()



cpi_with_unattributed = cpi.set_index("channel")[["spend_egp", "reported_installs", "cpi"]].copy()
cpi_with_unattributed["assumed_installs"] = cpi_with_unattributed["reported_installs"]
cpi_with_unattributed.loc["Google Ads", "assumed_installs"] += n_unattr
cpi_with_unattributed["recalculated_cpi"] = (
    cpi_with_unattributed["spend_egp"] / cpi_with_unattributed["assumed_installs"]
)

cpi_ranked = cpi_with_unattributed.sort_values("recalculated_cpi", ascending=False)
print(f"\n── CPI if all {n_unattr:,} unattributed signups count as extra Google Ads installs ──")
display(cpi_ranked.rename(columns={"cpi": "reported_cpi"}).round(2))
print(f"Highest recalculated CPI: {cpi_ranked.index[0]} ({cpi_ranked.iloc[0]['recalculated_cpi']:.2f} EGP)")


Clean users: 6,200

── Cost per signup ──

Other channels     : 1,256, ['Organic', 'Referral', 'Direct']
Unattributed       : 764


── CPI if all 764 unattributed signups count as extra Google Ads installs ──
Highest recalculated CPI: Google Ads (65.65 EGP)


,spend_egp,paid_signups,cost_per_signup
channel,,,
Meta,"145,369.33",2100,69.22
Google Ads,"168,517.02",1430,117.84
TikTok,"42,652.36",650,65.62


,spend_egp,reported_installs,reported_cpi,assumed_installs,recalculated_cpi
channel,,,,,
Google Ads,"168,517.02",1803,93.46,2567,65.65
Meta,"145,369.33",3355,43.33,3355,43.33
TikTok,"42,652.36",1185,35.99,1185,35.99


In [58]:
# =====================================================================
# PHASE 3 — Saturation quartiles + CPI-only split (45/35/20)
# =====================================================================
q3_daily = q3_spend.copy()
q3_daily["cpi_daily"] = q3_daily.spend_egp / q3_daily.platform_reported_installs


def classify(cpi_increase_pct):
    if cpi_increase_pct < SATURATION_FLAT_PCT:
        return "Flat CPI in Q3"
    if cpi_increase_pct < SATURATION_MODERATE_PCT:
        return "Moderate CPI increase"
    return "Large CPI increase"


# ── 1. Observed daily medians in the lowest and highest spend quartiles ──
quartile_rows = []
quartile_labels = ["Lowest", "Low-mid", "High-mid", "Highest"]

for channel in PAID_CHANNELS:
    daily = q3_daily.loc[q3_daily.channel.eq(channel)].copy()
    daily["spend_quartile"] = pd.qcut(
        daily.spend_egp, 4, labels=quartile_labels
    )

    medians = daily.groupby("spend_quartile", observed=True).agg(
        spend_egp=("spend_egp", "median"),
        installs=("platform_reported_installs", "median"),
        cpi=("cpi_daily", "median"),
    )
    low = medians.loc["Lowest"]
    high = medians.loc["Highest"]
    cpi_increase_pct = 100 * (high["cpi"] - low["cpi"]) / low["cpi"]

    quartile_rows.append({
        "channel": channel,
        "low_median_spend_egp": low["spend_egp"],
        "high_median_spend_egp": high["spend_egp"],
        "low_median_installs": low["installs"],
        "high_median_installs": high["installs"],
        "low_median_cpi": low["cpi"],
        "high_median_cpi": high["cpi"],
        "cpi_increase_pct": cpi_increase_pct,
        "state": classify(cpi_increase_pct),
    })

saturation = pd.DataFrame(quartile_rows).set_index("channel")

print("── Lowest vs highest spend quartile: median daily metrics ──")
display(saturation.round(2))


# ── 2. Actual Q3 allocation and CPI ──
q3_allocation = (
    q3_daily.groupby("channel")
    .agg(
        spend_egp=("spend_egp", "sum"),
        installs=("platform_reported_installs", "sum"),
    )
    .reindex(PAID_CHANNELS)
)

q3_total_spend = q3_allocation.spend_egp.sum()
q3_shares = (q3_allocation.spend_egp / q3_total_spend).to_dict()
q3_allocation.insert(1, "spend_split_pct", 100 * pd.Series(q3_shares))
q3_allocation["cpi"] = q3_allocation.spend_egp / q3_allocation.installs

q3_total_installs = q3_allocation.installs.sum()
q3_blended_cpi = q3_total_spend / q3_total_installs

print("\n── Actual Q3 allocation ──")
display(q3_allocation.round(2))


# ── 3. CPI-only allocation at the same total Q3 spend ──
comparison_shares = CPI_OPTIMISED_SHARES
recommended_shares = pd.Series(comparison_shares).reindex(PAID_CHANNELS)
recommended_spend = q3_total_spend * recommended_shares

recommended_allocation = pd.DataFrame({
    "spend_egp": recommended_spend,
    "spend_split_pct": 100 * recommended_shares,
    "installs": recommended_spend / q3_allocation.cpi,
    "cpi": q3_allocation.cpi,
})

recommended_total_spend = recommended_allocation.spend_egp.sum()
recommended_total_installs = recommended_allocation.installs.sum()
recommended_blended_cpi = recommended_total_spend / recommended_total_installs

print("\n── CPI-only allocation at total Q3 spend (modeled) ──")
display(recommended_allocation.round({
    "spend_egp": 2,
    "spend_split_pct": 2,
    "installs": 0,
    "cpi": 2,
}))


# ── 4. Blended CPI comparison at the same Q3 budget ──
split_comparison = pd.DataFrame({
    "total_spend_egp": [q3_total_spend, recommended_total_spend],
    "total_installs": [q3_total_installs, recommended_total_installs],
    "blended_cpi": [q3_blended_cpi, recommended_blended_cpi],
}, index=["Actual Q3 split", "CPI-only 45/35/20 (modeled)"])

split_comparison["cpi_reduction_vs_q3_egp"] = (
    q3_blended_cpi - split_comparison.blended_cpi
)
split_comparison["cpi_reduction_vs_q3_pct"] = (
    100 * split_comparison.cpi_reduction_vs_q3_egp / q3_blended_cpi
)

print("\n── Blended CPI comparison at the same Q3 budget ──")
display(split_comparison.round({
    "total_spend_egp": 2,
    "total_installs": 0,
    "blended_cpi": 2,
    "cpi_reduction_vs_q3_egp": 2,
    "cpi_reduction_vs_q3_pct": 2,
}))


# ── 5. CPI-only allocation at the December budget ──
cpi_split = pd.DataFrame({
    "q3_share_pct": {ch: round(100 * q3_shares[ch], 1) for ch in PAID_CHANNELS},
    "cpi_split_pct": {ch: round(100 * CPI_OPTIMISED_SHARES[ch], 1) for ch in PAID_CHANNELS},
    "cpi_split_egp": {ch: round(DEC_BUDGET * CPI_OPTIMISED_SHARES[ch], 0) for ch in PAID_CHANNELS},
})

print("\n── CPI-only December allocation (45/35/20) ──")
display(cpi_split)

── Lowest vs highest spend quartile: median daily metrics ──


,low_median_spend_egp,high_median_spend_egp,low_median_installs,high_median_installs,low_median_cpi,high_median_cpi,cpi_increase_pct,state
channel,,,,,,,,
Meta,786.00,"2,696.15",17.00,58.00,42.85,43.63,1.82,Flat CPI in Q3
Google Ads,"1,213.62","2,418.07",13.00,26.00,84.11,98.57,17.20,Large CPI increase
TikTok,213.65,739.46,5.00,21.00,34.93,37.06,6.09,Moderate CPI increase



── Actual Q3 allocation ──


,spend_egp,spend_split_pct,installs,cpi
channel,,,,
Meta,"145,369.33",40.77,3355,43.33
Google Ads,"168,517.02",47.26,1803,93.46
TikTok,"42,652.36",11.96,1185,35.99



── CPI-only allocation at total Q3 spend (modeled) ──


,spend_egp,spend_split_pct,installs,cpi
Meta,"160,442.42",45.00,"3,703.00",43.33
Google Ads,"124,788.55",35.00,"1,335.00",93.46
TikTok,"71,307.74",20.00,"1,981.00",35.99



── Blended CPI comparison at the same Q3 budget ──


,total_spend_egp,total_installs,blended_cpi,cpi_reduction_vs_q3_egp,cpi_reduction_vs_q3_pct
Actual Q3 split,"356,538.71","6,343.00",56.21,0.00,0.00
CPI-only 45/35/20 (modeled),"356,538.71","7,019.00",50.80,5.41,9.63



── CPI-only December allocation (45/35/20) ──


,q3_share_pct,cpi_split_pct,cpi_split_egp
Meta,40.80,45.00,"67,500.00"
Google Ads,47.30,35.00,"52,500.00"
TikTok,12.00,20.00,"30,000.00"


In [60]:
# =====================================================================
# PHASE 4 — Cost per activated user (README definition)
# Activated = KYC verified + ≥1 successful deposit within 7 days of signup
# =====================================================================
txn = pd.read_csv(DATA_DIR / "transactions.csv", parse_dates=["txn_ts"])

txn_clean = (
    txn
    .query("status == 'success' and txn_type == 'deposit'")
    .merge(
        users[["user_id", "signup_ts", "kyc_verified_ts", "channel"]],
        on="user_id",
        how="inner",
    )
)
txn_clean["days_since"] = (
    txn_clean.txn_ts - txn_clean.signup_ts
).dt.total_seconds() / 86400

activated_users = (
    txn_clean
    .query("days_since >= 0 and days_since <= 7")
    .query("kyc_verified_ts.notna()")
    .drop_duplicates("user_id")
)

paid_activated = (
    activated_users[activated_users.channel.isin(PAID_CHANNELS)]
    .groupby("channel").size()
    .reindex(PAID_CHANNELS).fillna(0).astype(int)
)

cpa = pd.DataFrame({
    "spend_egp": spend_by_channel,
    "signups": paid_signups,
    "activated_users": paid_activated,
    "activation_rate_pct": (100 * paid_activated / paid_signups).round(1),
    "cost_per_activated": (spend_by_channel / paid_activated).round(2),
})

print(f"Activated users (KYC + deposit ≤7d): {len(activated_users):,}\n")

print("── Reported CPI and cost per activated user ──")
display(
    cpa.join(
        cpi.set_index("channel")[["cpi"]].rename(columns={"cpi": "reported_cpi"})
    )
)

COST_PER_ACTIVATED = cpa["cost_per_activated"].to_dict()

Activated users (KYC + deposit ≤7d): 2,361

── Reported CPI and cost per activated user ──


,spend_egp,signups,activated_users,activation_rate_pct,cost_per_activated,reported_cpi
channel,,,,,,
Meta,"145,369.33",2100,522,24.90,278.49,43.33
Google Ads,"168,517.02",1430,730,51.00,230.85,93.46
TikTok,"42,652.36",650,132,20.30,323.12,35.99


In [61]:
# =====================================================================
# PHASE 5 — Budget splits compared on CPI and cost per activation
# =====================================================================
channel_cpi = q3_allocation["cpi"].reindex(PAID_CHANNELS)
channel_activation_cost = (
    cpa["spend_egp"] / cpa["activated_users"]
).reindex(PAID_CHANNELS)


def model_allocation(shares, budget):
    shares = pd.Series(shares).reindex(PAID_CHANNELS)
    spend = budget * shares

    allocation = pd.DataFrame({
        "spend_egp": spend,
        "spend_split_pct": 100 * shares,
        "installs": spend / channel_cpi,
        "activated_users": spend / channel_activation_cost,
    })
    allocation.loc["Blended"] = allocation.sum()
    allocation["cpi"] = allocation.spend_egp / allocation.installs
    allocation["cost_per_activated"] = (
        allocation.spend_egp / allocation.activated_users
    )
    return allocation


volume_rounding = {"installs": 0, "activated_users": 0}

# Compare actual Q3 with the recommended split at the same Q3 spend.
q3_budget_comparison = pd.concat({
    "Actual Q3 allocation": model_allocation(q3_shares, q3_total_spend),
    "Recommended 45/40/15 (modeled)": model_allocation(FINAL_SHARES, q3_total_spend),
}, names=["allocation", "channel"])

print("── Actual and recommended allocations at total Q3 spend ──")
display(q3_budget_comparison.round(2).round(volume_rounding))

# Compare both proposed splits and the Q3 mix at the December budget.
december_costs = pd.DataFrame({
    "Q3 shares": model_allocation(q3_shares, DEC_BUDGET).loc["Blended"],
    "CPI-only 45/35/20": model_allocation(
        CPI_OPTIMISED_SHARES, DEC_BUDGET
    ).loc["Blended"],
    "Recommended 45/40/15": model_allocation(
        FINAL_SHARES, DEC_BUDGET
    ).loc["Blended"],
}).T.drop(columns="spend_split_pct")

print("\n── December split comparison at the same 150,000 EGP budget ──")
display(december_costs.round(2).round(volume_rounding))

── Actual and recommended allocations at total Q3 spend ──


spend_egp  spend_split_pct  installs  activated_users   cpi  cost_per_activated
allocation                     channel                                                                                    
Actual Q3 allocation           Meta       145,369.33            40.77  3,355.00           522.00 43.33              278.49
                               Google Ads 168,517.02            47.26  1,803.00           730.00 93.46              230.85
                               TikTok      42,652.36            11.96  1,185.00           132.00 35.99              323.12
                               Blended    356,538.71           100.00  6,343.00         1,384.00 56.21              257.61
Recommended 45/40/15 (modeled) Meta       160,442.42            45.00  3,703.00           576.00 43.33              278.49
                               Google Ads 142,615.48            40.00  1,526.00           618.00 93.46              230.85
                               TikTok      53,480.81            15.00  1,486.00           166.00 35.99              323.12
                               Blended    356,538.71           100.00  6,715.00         1,359.00 53.10              262.27


── December split comparison at the same 150,000 EGP budget ──


,spend_egp,installs,activated_users,cpi,cost_per_activated
Q3 shares,"150,000.00","2,669.00",582.00,56.21,257.61
CPI-only 45/35/20,"150,000.00","2,953.00",563.00,50.80,266.59
Recommended 45/40/15,"150,000.00","2,825.00",572.00,53.10,262.27


# Performance Marketing | Q3 findings and December budget context

## Task 1 — Why I handled Marketing first

Performance Marketing needed an answer by Tuesday to lock the 150,000 EGP December
budget. That was the earliest deadline, ahead of the Thursday board pack, the Sunday
CVM request, and the loan-limit review due next month. The immediate decision was
whether Meta’s lower reported CPI justified moving most of Google’s budget to Meta.

I checked the reported CPI first, then compared the channels on internal signups
and activation before drawing a budget conclusion. Deadline drove the order of
work. Impact and downstream dependencies would break ties between requests with
similar deadlines.

## Task 2 — Data quality, handling, and assumptions

| Issue | What I found | How I handled it |
|---|---|---|
| QA users | 30 accounts matched the reviewed QA names, including `تجربة`. | Excluded the listed accounts from signup counts and verified that none of their transactions reached the activation input. The check covers the QA accounts identified by name. |
| Duplicate transactions | 577 duplicate `txn_id` rows. | Kept one row per transaction ID before auditing user matches and calculating activation. |
| QA transactions and true orphans | 682 unique transactions belonged to listed QA users. Zero transactions were missing a match in the raw users table. | Audited against raw user IDs. The earlier “orphan” count came from checking against the users table after its QA rows had been removed. Task 2 excludes those 682 transactions in the shared export. the source of this duplication must be identified and addressed i didnt have enough to find out why operationally |
| Transaction confirmation | The extract records a transaction status and a user ID. | Used successful deposits linked to users in the extract, after deduplication and QA exclusion. Treated a success status and a user match as the available confirmation. |
| Inconsistent attribution labels | The raw `utm_source` values use several names for the same channels. | Trimmed and lowercased the source. Mapped Meta, Facebook and Instagram aliases to Meta, Google and AdWords aliases to Google Ads, and TikTok aliases to TikTok. Grouped Organic, Referral and Direct under Other channels for reporting. |
| Blank attribution | 764 signups had no usable source after QA removal. | Kept them labeled Unattributed. Assumed they were organic-like in the base case and excluded them from paid-channel denominators. Tested an alternative that gives them all to Google. |

The signup counts reconcile to **6,200** after QA removal: **2,100 Meta + 1,430
Google Ads + 650 TikTok + 1,256 Other channels + 764 Unattributed**.

## Task 3 — Marketing analysis and recommendation

### What changes when we look beyond CPI

Q3 spend was **356,538.71 EGP** for **6,343 platform-reported installs**, giving a
blended CPI of **56.21 EGP**. Cost per signup uses internally recorded paid
signups. Cost per activation uses paid signups who were KYC verified and made
at least one successful deposit within seven days of signup, per the dataset
README. Activation counts each qualifying user once.

| Channel | Reported CPI (EGP) | Cost per signup (EGP) | Activated users | Activation rate | Cost per activated user (EGP) |
|---|---:|---:|---:|---:|---:|
| Meta | 43.33 | 69.22 | 522 | 24.9% | 278.49 |
| Google Ads | 93.46 | 117.84 | 730 | 51.0% | 230.85 |
| TikTok | 35.99 | 65.62 | 132 | 20.3% | 323.12 |

Meta’s CPI advantage over Google is real in the platform data. Google has the
highest reported CPI and the lowest cost per activated user. About half of
Google’s attributed signups activated, compared with about a quarter of Meta’s.
TikTok had the lowest install cost and the highest activation cost.

### What if all unattributed signups came from Google?

For the hypothetical CPI calculation, assume each of the 764 unattributed
signups came from a separate Google-driven install missing from Google’s reported
count. That raises Google’s install denominator from 1,803 to 2,567 and reduces
its hypothetical CPI from **93.46 to 65.65 EGP**. Google still has the highest
CPI, ahead of Meta at **43.33 EGP** and TikTok at **35.99 EGP**.

The user table contains missing signup attribution. Some of those installs may
already be included in Google’s reported count. The hypothetical CPI is a
deliberately generous scenario. Use 93.46 EGP as the reported Q3 figure.

### What higher-spend days tell us

The notebook compares median daily spend, reported installs, and CPI in each
channel’s lowest and highest spend quartiles. Meta’s median daily CPI rose
**1.8%**, Google’s **17.2%**, and TikTok’s **6.1%**. The spend and install
medians show the volume behind each comparison.

These are observed differences across days. Campaign mix and timing may account
for some of the increase. The quartiles measure install costs, and the analysis
has yet to establish how cost per activation responds to spend.

### December budget decision

| Allocation (Meta / Google / TikTok) | Modeled installs | Modeled activations | Blended CPI (EGP) | Cost per activation (EGP) |
|---|---:|---:|---:|---:|
| Q3 shares (40.77% / 47.26% / 11.96%) | 2,669 | 582 | 56.21 | 257.61 |
| CPI-only (45% / 35% / 20%) | 2,953 | 563 | 50.80 | 266.59 |
| Balanced proposal (45% / 40% / 15%) | 2,825 | 572 | 53.10 | 262.27 |

All three December scenarios use the same 150,000 EGP budget. The model applies
observed Q3 channel unit costs to each allocation and calculates blended costs
from total spend and modeled outcomes. It holds channel unit costs constant.

**45/40/15 is the balanced proposal**: 67,500 EGP Meta, 60,000 EGP Google Ads,
and 22,500 EGP TikTok. Compared with 45/35/20, it buys about nine more activations
with a higher blended CPI. Compared with keeping Q3 shares, it buys about 156 more
installs and produces about ten fewer activations. Keeping Q3 shares gives the
lowest modeled cost per activation of the three. The decision turns on whether
December prioritizes activation volume alone or also values lower CPI and a
smaller share of spend concentrated in Google.

The Q3 budget replay illustrates the same trade-off at actual Q3 spend. At
356,538.71 EGP, **45/40/15** would model **6,715 installs and 1,359 activations**
versus **6,343 installs and 1,384 activations** under the actual allocation.
Blended CPI falls from **56.21 to 53.10 EGP** and cost per activation rises from
**257.61 to 262.27 EGP**.

## Task 5 — Context for the final communications

- **Performance Marketing Slack reply:** The team’s Meta versus Google CPI
  figures check out. Google’s advantage appears further down the funnel, at
  activation. The reply needs to connect that finding to the allocation decision
  and explain the trade-off in plain language.
- **CPGO one-pager:** The marketing takeaway is that cheaper installs can coexist
  with more expensive activations. Keep this as one candidate finding until the
  KPI, CVM, and credit analyses are ready and the cross-case decision is clear.

## Task 6 — Context for the final follow-up

Open questions for the final write-up include which unattributed signups came
from paid channels, how many additional activations each channel produces when
spend changes, and how much revenue those activated users generate. The Q3
quartiles can inform a spend test. A controlled comparison would help estimate
incremental activations and cost per incremental activation.

If time allows, extend the notebook to show **average revenue per activated
user** and **total revenue versus ad spend** by platform. Use a common
post-signup observation window and the revenue definitions in the dataset README.